# Clinical Mortality and LOS Pipeline

This notebook demonstrates the leakage-safe preprocessing workflow used in this project. The key correction is simple but important: split patients first, then fit imputation and encoding only on the training split.

## 1. Setup

Place the local admissions CSV at `data/raw/unified_patient_admissions_ccs.csv`. Patient-level data is ignored by git, so the notebook is safe to keep in a public repository.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess_split_first import SplitConfig, build_splits

RAW_CSV = PROJECT_ROOT / 'data/raw/unified_patient_admissions_ccs.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data/processed'
RAW_CSV.exists()

## 2. Build Leakage-Safe Splits

The pipeline groups by `subject_id`, fits preprocessing on train only, and saves both transformed matrices and raw split files for auditability.

In [ ]:
if not RAW_CSV.exists():
    raise FileNotFoundError(f'Add your local source CSV here: {RAW_CSV}')

audit = build_splits(
    input_csv=RAW_CSV,
    output_dir=OUTPUT_DIR,
    config=SplitConfig(validation_size=0.15, test_size=0.15, random_state=42),
)
audit['split_summary']

## 3. Review the Audit

The audit records the fit policy, excluded leakage columns, feature counts, and high-cardinality categoricals that were intentionally dropped.

In [ ]:
audit_path = OUTPUT_DIR / 'preprocessing_audit.json'
audit = json.loads(audit_path.read_text())
print(json.dumps({
    'fit_policy': audit['fit_policy'],
    'excluded_from_features': audit['excluded_from_features'],
    'numeric_feature_count': audit['numeric_feature_count'],
    'categorical_feature_count': audit['categorical_feature_count'],
}, indent=2))

## 4. Inspect Output Matrices

These matrices are ready for downstream models such as logistic regression, XGBoost, neural sequence models, or transformer experiments.

In [ ]:
split_shapes = {}
for split in ['train', 'val', 'test']:
    matrix = pd.read_csv(OUTPUT_DIR / f'{split}_model_matrix.csv')
    split_shapes[split] = matrix.shape
split_shapes

## 5. Leakage Checks

The raw split files make it easy to verify that no patient appears in more than one split.

In [ ]:
subject_sets = {
    split: set(pd.read_csv(OUTPUT_DIR / f'{split}_raw_split.csv')['subject_id'])
    for split in ['train', 'val', 'test']
}

assert subject_sets['train'].isdisjoint(subject_sets['val'])
assert subject_sets['train'].isdisjoint(subject_sets['test'])
assert subject_sets['val'].isdisjoint(subject_sets['test'])

'No subject leakage across splits.'